In [ ]:
from io import BytesIO
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:


source_path = Path("SI-dataset.xlsx")  # Path to the original SI workbook.
input_path = Path("derived/intermediate") / "dataset.xlsx"  # Path to the input workbook.
output_path = Path("derived/intermediate") / "prepared_dataset.xlsx"  # Path to the generated workbook.
point_id = "Data point number"  # Unique experimental-record identifier column.
method_column = "Cu characterization method"  # Method column.
calc_columns = ["Calc T", "Calc D"]  # Calcination temperature and duration columns.
test_fraction = 0.2  # Test fraction.
split_seed = 0  # Split seed.

In [ ]:
data = pd.read_excel(input_path, sheet_name="data", engine="openpyxl")  # Input records for this analysis.
methods = pd.read_excel(  # Synthesis method categories or their codes.
    source_path,
    sheet_name="data",
    usecols=["Number", "N2O Method"],
    engine="openpyxl",
).rename(columns={"Number": point_id, "N2O Method": method_column})

data = data.merge(methods, on=point_id, how="left", validate="one_to_one")
method = data[method_column].astype("string").str.strip()  
if len(data) != len(methods) or method.fillna("").eq("").any():
    raise ValueError("Match the source workbook to the generated input and check the method labels.")

duplicate_point = 231  
retained_point = 232  
match_columns = ["Catalyst number", "Reference number", "GHSV", "H2/CO2", "P", "T"]  
by_point = data.set_index(point_id)  
if not by_point.loc[duplicate_point, match_columns].equals(
    by_point.loc[retained_point, match_columns]
):
    raise ValueError("Review the suspected duplicate before excluding it.")

has_n2o = method.str.contains(r"N\s*[2₂]\s*O", case=False, regex=True)  
has_h2 = method.str.contains(r"H\s*[2₂]", case=False, regex=True)  
has_xrd = method.str.contains("XRD", case=False, regex=False)  

reason = pd.Series(pd.NA, index=data.index, dtype="string")  
reason.loc[~has_n2o] = "Other Cu characterization method"
reason.loc[~has_n2o & has_h2] = "H2 chemisorption"
reason.loc[~has_n2o & has_xrd] = "XRD-derived Cu descriptors"
reason.loc[data[point_id].eq(duplicate_point)] = f"Suspected duplicate of point {retained_point}"

screened = data.loc[reason.isna()].copy()  
excluded = data.loc[reason.notna()].assign(**{"Exclusion reason": reason.loc[reason.notna()]})  
screened[calc_columns] = screened[calc_columns].apply(pd.to_numeric)
print(f"Input records: {len(data)}")
print(f"N2O-labelled records after screening: {len(screened)}")
print(excluded["Exclusion reason"].value_counts().to_string())

In [ ]:
means = screened[calc_columns].mean()  # Calculated means for imputation or plotting.
if means.isna().any():
    raise ValueError("Each calcination column needs observed values.")

fill_values = (means + 0.5).astype(int)  
filled = screened.copy()  

for column in calc_columns:  # Current column name.
    filled[f"{column} imputed"] = filled[column].isna()

filled[calc_columns] = filled[calc_columns].fillna(fill_values)

In [ ]:
if output_path.exists():
    raise FileExistsError(f"Choose a new output path: {output_path.resolve()}")

with BytesIO() as contents:  # In-memory Excel workbook bytes.
    with pd.ExcelWriter(contents, engine="openpyxl") as writer:  # Excel writer for the result workbook.
        filled.to_excel(writer, sheet_name="data", index=False)
        excluded.to_excel(writer, sheet_name="excluded", index=False)

    with output_path.open("xb") as handle:  # Open file handle.
        handle.write(contents.getbuffer())